# Hybrid Analyzer — SLM & Edge AI Enrichment

**Pipeline:**
```
raw JSON (4 sources) → Clean & Deduplicate → Embedding Classification → Ollama Summary → enriched JSON
```

**Source-specific challenges handled:**
| Source | Issue | Fix |
|---|---|---|
| `medium` | Duplicates (title=subtitle), truncated summaries (…) | Dedup by URL, use title as fallback |
| `linkedin` | Summary always starts with title (redundant), 69/92 unique URLs | Dedup + strip title from summary |
| `semantic_scholar` | Full abstracts (up to 2500 chars), truncated mid-sentence | Trim to 500 chars at sentence boundary |
| `arxiv` | Same format as semantic_scholar but richer metadata | Same treatment |


## Cell 0 — Installs & Imports

In [21]:
import json
import re
import time
import numpy as np
from tqdm import tqdm
from pathlib import Path
from collections import Counter

import ollama
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("✅ All imports OK")

✅ All imports OK


## Cell 1 — Load All Sources

Point `SOURCE_FILES` to your JSON files. Each file can contain a single source or a mix.

In [10]:
# ── CONFIGURE YOUR FILE PATHS HERE ────────────────────────────────────────────
SOURCE_FILES = [
    "../data/raw_articles.json"                         # arxiv — add when available
]

# ── LOAD ALL FILES ─────────────────────────────────────────────────────────────
raw_all = []
for path in SOURCE_FILES:
    p = Path(path)
    if not p.exists():
        print(f"⚠️  File not found, skipping: {path}")
        continue
    with open(p, "r", encoding="utf-8") as f:
        data = json.load(f)
    raw_all.extend(data)
    print(f"Loaded {len(data):>4} entries from {p.name}")

print(f"\nTotal raw entries: {len(raw_all)}")

# ── SOURCE BREAKDOWN ──────────────────────────────────────────────────────────
src_counts = Counter(a.get("source", "unknown") for a in raw_all)
print("\nEntries per source:")
for src, count in src_counts.items():
    print(f"  {src:<20} {count}")

Loaded  823 entries from raw_articles.json

Total raw entries: 823

Entries per source:
  google_scholar       233
  medium               94
  linkedin             92
  arxiv                254
  semantic_scholar     150


## Cell 2 — Source-Aware Cleaning

Each source has specific issues. We handle them with **source-specific normalizers** 
that all produce the same clean output format.

**What each normalizer does:**
- `clean_medium()` → removes duplicate subtitle entries, cleans truncation artifacts
- `clean_linkedin()` → deduplicates by URL, removes title repetition in body, trims to key content
- `clean_semantic_scholar()` → trims long abstracts cleanly at sentence boundaries
- `clean_arxiv()` → same as semantic_scholar (identical schema)


In [11]:
# ═══════════════════════════════════════════════════════════════════════════════
# SOURCE-SPECIFIC CLEANERS
# Each returns a cleaned article dict, or None to discard the entry.
# ═══════════════════════════════════════════════════════════════════════════════

def trim_at_sentence(text: str, max_chars: int = 500) -> str:
    """Trim text to max_chars, but always break at a sentence boundary."""
    if len(text) <= max_chars:
        return text
    trimmed = text[:max_chars]
    # Find last sentence-ending punctuation before the limit
    last_stop = max(trimmed.rfind('. '), trimmed.rfind('! '), trimmed.rfind('? '))
    if last_stop > max_chars * 0.5:  # only cut if we keep at least 50% of content
        return trimmed[:last_stop + 1].strip()
    return trimmed.strip() + "..."


def clean_medium(article: dict) -> dict | None:
    """
    Medium-specific issues:
    1. Duplicate entries: scraper returned both the title and the subtitle
       as separate articles with the same URL.
       Fix: handled at deduplication stage (by URL).
    2. Summaries are truncated Medium previews ending with '…'
       Fix: strip the ellipsis, use title as primary content if summary is weak.
    3. Some summaries ARE the title (content-free)
       Fix: discard if summary == title or summary is too short.
    """
    title   = article.get("title", "").strip()
    summary = article.get("summary", "").strip()

    # Remove Medium truncation artifacts
    summary = summary.rstrip("…").rstrip(".").strip()

    # Discard if summary is empty or is just the title repeated
    if not summary or summary.lower() == title.lower():
        summary = ""  # will use title-only input_text

    # Discard if the entry has essentially no content at all
    if not title:
        return None

    article["title"]   = title
    article["summary"] = summary
    return article


def clean_linkedin(article: dict) -> dict | None:
    """
    LinkedIn-specific issues:
    1. Summary always starts with the full title text (100% of entries)
       Fix: strip the title prefix from the summary body.
    2. Multiple posts from same author/company have same URL
       Fix: handled at deduplication stage.
    3. Summaries end with '… plus' or '… see more' (LinkedIn UI artifact)
       Fix: strip those suffixes.
    4. Content is conversational/marketing, not technical
       Fix: lower the classification confidence threshold before using Ollama
       (handled in Cell 4 with a source-specific prompt).
    """
    title   = article.get("title", "").strip()
    summary = article.get("summary", "").strip()

    # Strip title prefix from summary (LinkedIn always includes it)
    if summary.startswith(title):
        summary = summary[len(title):].strip()
        # Clean leading punctuation left over
        summary = summary.lstrip(".\n\r").strip()

    # Strip LinkedIn trailing artifacts
    for artifact in ["… plus", "…plus", "… see more", "…see more", "…"]:
        if summary.endswith(artifact):
            summary = summary[: -len(artifact)].strip()

    # Trim to a reasonable length
    summary = trim_at_sentence(summary, max_chars=400)

    if not title:
        return None

    article["title"]   = title
    article["summary"] = summary
    return article


def clean_semantic_scholar(article: dict) -> dict | None:
    """
    Semantic Scholar-specific issues:
    1. Abstracts are very long (up to 2500 chars) — too much context for bge-small
       Fix: trim to 500 chars at a sentence boundary.
    2. Some abstracts are cut mid-sentence by the API
       Fix: the trim_at_sentence() already handles this.
    3. Multiple authors (up to 6+) — no changes needed for classification.
    """
    title   = article.get("title", "").strip()
    summary = article.get("summary", "").strip()

    # Trim long abstracts to avoid overwhelming the embedding model
    # bge-small has a 512-token limit anyway, so anything beyond ~380 words is truncated
    summary = trim_at_sentence(summary, max_chars=500)

    if not title:
        return None

    article["title"]   = title
    article["summary"] = summary
    return article


def clean_arxiv(article: dict) -> dict | None:
    """
    arXiv uses the same schema as Semantic Scholar.
    Additional: arXiv titles sometimes include version tags like [v2]
    """
    title = article.get("title", "").strip()
    # Remove version tags
    title = re.sub(r'\[v\d+\]', '', title).strip()
    article["title"] = title
    return clean_semantic_scholar(article)


# ── DISPATCH MAP ──────────────────────────────────────────────────────────────
# Maps source name → cleaner function
CLEANERS = {
    "medium":           clean_medium,
    "linkedin":         clean_linkedin,
    "semantic_scholar": clean_semantic_scholar,
    "arxiv":            clean_arxiv,
}

print("✅ Cleaner functions defined")
print(f"   Registered sources: {list(CLEANERS.keys())}")

✅ Cleaner functions defined
   Registered sources: ['medium', 'linkedin', 'semantic_scholar', 'arxiv']


## Cell 3 — Deduplicate & Apply Cleaners

Two-stage deduplication:
1. **By URL** — catches Medium duplicates and LinkedIn same-URL posts
2. **By normalized title** — catches cross-source duplicates (same paper on arxiv AND semantic_scholar)

In [12]:
def normalize_title(title: str) -> str:
    """Lowercase, remove punctuation/spaces — used for cross-source dedup."""
    return re.sub(r'[^a-z0-9]', '', title.lower())


# ── STAGE 1: Apply source-specific cleaners ────────────────────────────────────
cleaned = []
skipped_no_cleaner = []

for art in raw_all:
    source = art.get("source", "unknown")
    cleaner = CLEANERS.get(source)

    if cleaner is None:
        # Unknown source: apply minimal cleaning, don't discard
        skipped_no_cleaner.append(source)
        art["summary"] = trim_at_sentence(art.get("summary", ""), 500)
        cleaned.append(art)
    else:
        result = cleaner(dict(art))  # work on a copy
        if result is not None:
            cleaned.append(result)

print(f"After cleaning:  {len(cleaned)} / {len(raw_all)} entries kept")
if skipped_no_cleaner:
    print(f"⚠️  Unknown sources (no cleaner): {set(skipped_no_cleaner)}")

# ── STAGE 2: Deduplicate by URL ────────────────────────────────────────────────
seen_urls = set()
deduped_url = []
for art in cleaned:
    url = art.get("url", "").split("?")[0]  # strip query params
    if url not in seen_urls:
        seen_urls.add(url)
        deduped_url.append(art)

print(f"After URL dedup: {len(deduped_url)} entries")

# ── STAGE 3: Deduplicate by normalized title (cross-source) ───────────────────
seen_titles = set()
articles = []
for art in deduped_url:
    norm = normalize_title(art.get("title", ""))
    if norm and norm not in seen_titles:
        seen_titles.add(norm)
        articles.append(art)

print(f"After title dedup: {len(articles)} unique articles")

# ── BUILD input_text ──────────────────────────────────────────────────────────
# This is the single text field fed to both embedding model and Ollama.
# Strategy per source:
#   medium          → title (summary too short/truncated to add value)
#   linkedin        → title + cleaned summary body (now non-redundant)
#   semantic_scholar → title + trimmed abstract (rich, high-quality)
#   arxiv           → same as semantic_scholar

SOURCE_INPUT_STRATEGY = {
    "medium":           lambda t, s: t if not s else f"{t}. {s}",
    "linkedin":         lambda t, s: f"{t}. {s}" if s else t,
    "semantic_scholar": lambda t, s: f"{t}. {s}" if s else t,
    "arxiv":            lambda t, s: f"{t}. {s}" if s else t,
}

for art in articles:
    source   = art.get("source", "unknown")
    title    = art.get("title", "").strip()
    summary  = art.get("summary", "").strip()
    strategy = SOURCE_INPUT_STRATEGY.get(source, lambda t, s: f"{t}. {s}" if s else t)
    art["input_text"] = strategy(title, summary)

# ── FINAL BREAKDOWN ────────────────────────────────────────────────────────────
print("\nFinal article count per source:")
for src, count in Counter(a.get("source") for a in articles).items():
    avg_len = int(np.mean([len(a['input_text']) for a in articles if a.get('source') == src]))
    print(f"  {src:<20} {count:>4} articles | avg input_text length: {avg_len} chars")

# ── PREVIEW ────────────────────────────────────────────────────────────────────
print("\n── Sample input_text per source ──")
shown = set()
for art in articles:
    src = art.get("source")
    if src not in shown:
        shown.add(src)
        print(f"\n[{src}]")
        print(art['input_text'][:200])

After cleaning:  823 / 823 entries kept
⚠️  Unknown sources (no cleaner): {'google_scholar'}
After URL dedup: 751 entries
After title dedup: 750 unique articles

Final article count per source:
  google_scholar        231 articles | avg input_text length: 266 chars
  medium                 47 articles | avg input_text length: 162 chars
  linkedin               69 articles | avg input_text length: 445 chars
  arxiv                 254 articles | avg input_text length: 485 chars
  semantic_scholar      149 articles | avg input_text length: 486 chars

── Sample input_text per source ──

[google_scholar]
CoDi-ABEM: An efficient LLM-based prompt compression and knowledge distillation fusion framework for automated multi-zone building energy modeling. compression and knowledge distillation for multi- li

[medium]
Microsoft Foundry Local. Foundry Local is an on-device AI inference solution that lets you run large AI models locally (on your own hardware) via a CLI, SDK, or

[linkedin]
Over the

## Cell 4 — Taxonomy Anchors & Embedding Classification

**Why anchors are source-agnostic:** The embedding model encodes *semantic meaning*, not surface patterns. 
Whether the article is a LinkedIn post or an arXiv abstract, "quantization" maps to the same region 
of the vector space. So one set of anchors works for all 4 sources.

**The only source-specific tuning:** LinkedIn articles are conversational/marketing — they tend to have 
lower confidence scores because their vocabulary is less technical. We flag those for review.

In [13]:
# ═══════════════════════════════════════════════════════════════════════════════
# LEVEL 1 — CATEGORY ANCHORS
# Enriched with keywords from the full survey taxonomy (§2-§6)
# Each description covers: what it is + technical terms + applications + challenges
# ═══════════════════════════════════════════════════════════════════════════════

ANCHORS = {

    "Data-level": (
        # § 2.2.1 Data Preprocessing + § 5.1 Data Optimization
        "Data optimization and preprocessing strategies for on-device AI and small language models. "
        "Includes data filtering, feature extraction, data aggregation, data quantization, "
        "edge computing frameworks for data pipelines. "
        # § 4.4 Communication Limitations + § 4.5 Data Privacy
        "Covers data preprocessing at the edge to reduce communication overhead, edge caching, "
        "data privacy protection, compliance, federated learning data strategies, "
        "data protection against security attacks. "
        # § 5.1 + § 6.2
        "Also includes dataset curation, synthetic data generation, data augmentation, "
        "corpus filtering, instruction tuning datasets, RLHF data, DPO data, "
        "benchmark datasets, training data efficiency, adaptive learning data pipelines, "
        "intelligent data management for continuous learning on device."
    ),

    "Model-level": (
        # § 2.1.2 AI Models + § 5.2 Model Optimization
        "Model compression, optimization, and design techniques for small and efficient AI models. "
        "Includes traditional machine learning methods adapted for edge, parameter sharing, "
        "weight pruning, model quantization (INT4, INT8, GGUF, AWQ, GPTQ, post-training quantization), "
        "knowledge distillation from large to small models, low-rank factorization, "
        "hardware-aware neural architecture search (NAS), energy-efficient model design. "
        # § 4.1 Limited Computational Resources
        "Addresses limited processing power, model complexity reduction, optimization algorithms "
        "for constrained devices. "
        # § 2.2.2 Model Development + compact architectures
        "Covers LoRA fine-tuning, adapter layers, compact architectures like Phi, Gemma, Mistral, "
        "TinyLlama, MobileLLM, MobileNet, efficient transformers. "
        # § 6.2 Adaptability
        "Also includes model adaptability, cross-device migration, continuous learning, "
        "adaptive learning, intelligent decision-making at model level, "
        "fairness and bias in model design, foundation models compression."
    ),

    "System-level": (
        # § 2.1.1 Edge Devices + § 2.1.3 On-Device vs Cloud + § 5.3 System Optimization
        "Hardware and software system infrastructure for on-device AI and edge intelligence. "
        "Includes software optimization (inference engines, runtime, compilers), "
        "hardware optimization (NPU, GPU, CPU-first inference, custom silicon, ONNX, TensorRT, llama.cpp). "
        # § 3 Applications
        "Covers deployment on edge devices: smartphones, IoT devices, smart homes, "
        "industrial automation, smart agriculture, autonomous driving, "
        "vehicle-to-everything (V2X), AR and VR, smart manufacturing. "
        # § 4.2 Storage + § 4.3 Energy
        "Addresses storage space limitations, memory usage, data management on device, "
        "battery life, dynamic energy management, hardware power optimization. "
        # § 3.3 Edge Computing + § 6.1 Emerging Technologies
        "Includes real-time data processing, intelligent traffic management, "
        "edge computing frameworks, TinyML deployment, Raspberry Pi, Arduino, mobile deployment, "
        "5G and beyond infrastructure, latency reduction, memory footprint, "
        # § 6.3 Green Computing + § 4.6 Transferability
        "green computing, sustainability, resource sharing, circular utilization, "
        "environmental monitoring, cross-device migration, system integration, "
        "production deployment pipelines at the edge."
    ),
}


# ═══════════════════════════════════════════════════════════════════════════════
# LEVEL 2 — SUBCATEGORY ANCHORS (one dict per category)
# Only computed for the winning category of each article
# ═══════════════════════════════════════════════════════════════════════════════

SUB_ANCHORS = {

    "Data-level": {
        # § 5.1 Data Optimization subcategories
        "Data Filtering":              "Selecting high-quality training samples, removing noise, deduplication, quality filtering of datasets, relevance scoring for pretraining corpora.",
        "Feature Extraction":          "Extracting compact representations from raw data, dimensionality reduction, embedding extraction, input compression before model inference.",
        "Data Aggregation":            "Combining data from multiple edge devices, federated data collection, distributed data pooling without centralizing raw data.",
        "Data Quantization":           "Quantizing input data or activations before processing, reducing data precision for transmission or storage at the edge.",
        "Edge Computing Frameworks":   "Frameworks and pipelines for processing data locally at the edge before cloud transmission, Apache Kafka, MQTT, edge data pipelines.",
        "Privacy & Data Security":     "Federated learning, differential privacy, data anonymization, on-device data protection, compliance, resistance to inference attacks.",
        "Synthetic & Augmented Data":  "Synthetic data generation, data augmentation, instruction tuning datasets, RLHF data, DPO data, distillation datasets from LLMs.",
    },

    "Model-level": {
        # § 5.2 Model Optimization subcategories
        "Pruning":                            "Removing redundant weights or neurons from neural networks, structured pruning, unstructured pruning, magnitude-based pruning, sparse models.",
        "Model Quantization":                 "Reducing numerical precision of model weights and activations: INT4, INT8, GGUF, AWQ, GPTQ, post-training quantization, quantization-aware training.",
        "Knowledge Distillation":             "Transferring knowledge from large teacher models to small student models, soft labels, feature distillation, task-specific distillation, LLM-to-SLM distillation.",
        "Low-rank Factorization":             "Decomposing weight matrices into low-rank approximations, SVD decomposition, LoRA, tensor decomposition for parameter reduction.",
        "Hardware-aware NAS":                 "Neural architecture search optimized for specific hardware constraints, latency-aware NAS, energy-aware architecture search, AutoML for edge.",
        "Energy-efficient Model Design":      "Designing models with low power consumption, battery-aware architectures, efficient transformers, MobileNet, MobileLLM, TinyLlama.",
        "Parameter Sharing":                  "Sharing weights across layers or heads, weight tying, cross-layer parameter reuse, compact model design through shared representations.",
        "Adaptability & Continuous Learning": "Continuous learning, domain adaptation, LoRA fine-tuning, adapter layers, cross-device model migration, personalization on device.",
    },

    "System-level": {
        # § 5.3 System Optimization + § 3 Applications + § 4 Challenges
        "Software Optimization":       "Inference engine optimization, compiler optimization, runtime efficiency, ONNX, TensorRT, llama.cpp, operator fusion, kernel optimization.",
        "Hardware Optimization":       "Custom silicon, NPU, GPU acceleration, CPU-first inference, hardware-software co-design, FPGA, chip design for AI workloads.",
        "Energy & Memory Management":  "Battery life optimization, dynamic energy management, memory footprint reduction, storage space management, RAM optimization on device.",
        "Edge & IoT Deployment":       "Deploying AI on edge devices: Raspberry Pi, Arduino, smartphones, IoT sensors, smart home, industrial automation, TinyML deployment.",
        "Autonomous & Smart Systems":  "Autonomous driving, vehicle-to-everything (V2X), path planning, environmental perception, intelligent traffic management, smart manufacturing, AR/VR.",
        "Privacy & Security Systems":  "System-level data protection, secure inference, model confidentiality, resistance to adversarial attacks, compliance frameworks at the edge.",
        "Green & Sustainable AI":      "Energy-efficient deployment, green computing, carbon footprint of AI, resource sharing, sustainability, environmental monitoring systems.",
    },
}


print("✅ Anchors defined")
print(f"   Level 1 categories: {list(ANCHORS.keys())}")
for cat, subs in SUB_ANCHORS.items():
    print(f"   Level 2 [{cat}]: {list(subs.keys())}")

✅ Anchors defined
   Level 1 categories: ['Data-level', 'Model-level', 'System-level']
   Level 2 [Data-level]: ['Data Filtering', 'Feature Extraction', 'Data Aggregation', 'Data Quantization', 'Edge Computing Frameworks', 'Privacy & Data Security', 'Synthetic & Augmented Data']
   Level 2 [Model-level]: ['Pruning', 'Model Quantization', 'Knowledge Distillation', 'Low-rank Factorization', 'Hardware-aware NAS', 'Energy-efficient Model Design', 'Parameter Sharing', 'Adaptability & Continuous Learning']
   Level 2 [System-level]: ['Software Optimization', 'Hardware Optimization', 'Energy & Memory Management', 'Edge & IoT Deployment', 'Autonomous & Smart Systems', 'Privacy & Security Systems', 'Green & Sustainable AI']


In [14]:
# ── LOAD MODEL ────────────────────────────────────────────────────────────────
import os, time, numpy as np
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("Loading BAAI/bge-small-en-v1.5...")
emb_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
print(f"✅ Model loaded (dim={emb_model.get_sentence_embedding_dimension()})")


def encode(texts: list[str]) -> np.ndarray:
    """Encode with bge prefix + L2 normalization."""
    prefixed = [f"Represent this sentence: {t}" for t in texts]
    return emb_model.encode(prefixed, normalize_embeddings=True, batch_size=32)


# ── ENCODE LEVEL 1 ANCHORS ────────────────────────────────────────────────────
cat_labels = list(ANCHORS.keys())
cat_embs   = encode(list(ANCHORS.values()))
print(f"\nLevel 1 anchors encoded: {cat_embs.shape}")

# ── ENCODE LEVEL 2 ANCHORS (one matrix per category) ─────────────────────────
sub_embs = {}
for cat, subs in SUB_ANCHORS.items():
    sub_labels = list(subs.keys())
    sub_vecs   = encode(list(subs.values()))
    sub_embs[cat] = {"labels": sub_labels, "embs": sub_vecs}
    print(f"  [{cat}] subcategory anchors: {sub_vecs.shape}")

# ── ENCODE ARTICLES ───────────────────────────────────────────────────────────
# 'articles' comes from Cell 3 (the cleaned + deduped list)
print(f"\nEncoding {len(articles)} articles...")
t0 = time.time()
article_embs = encode([a["input_text"] for a in articles])
print(f"✅ Encoded in {time.time()-t0:.1f}s")


# ── LEVEL 1 CLASSIFICATION ────────────────────────────────────────────────────
CONFIDENCE_THRESHOLDS = {
    "arxiv":            0.75,
    "semantic_scholar": 0.75,
    "google_scholar":   0.65,
    "medium":           0.68,
    "linkedin":         0.60,
    "default":          0.70,
}

cat_sim = cosine_similarity(article_embs, cat_embs)  # (n_articles, 3)

for i, art in enumerate(articles):
    scores    = cat_sim[i]
    best_idx  = int(np.argmax(scores))
    threshold = CONFIDENCE_THRESHOLDS.get(art.get("source"), CONFIDENCE_THRESHOLDS["default"])

    art["category"]            = cat_labels[best_idx]
    art["category_confidence"] = round(float(scores[best_idx]), 4)
    art["category_scores"]     = {
        label: round(float(scores[j]), 4)
        for j, label in enumerate(cat_labels)
    }
    art["needs_review"] = art["category_confidence"] < threshold


# ── LEVEL 2 SUBCATEGORY CLASSIFICATION ───────────────────────────────────────
# Only run against the subcategories of the winning category
for i, art in enumerate(articles):
    cat = art["category"]
    sub_info  = sub_embs[cat]
    sub_sim   = cosine_similarity(article_embs[i].reshape(1, -1), sub_info["embs"])[0]
    best_sub  = int(np.argmax(sub_sim))

    art["subcategory"]            = sub_info["labels"][best_sub]
    art["subcategory_confidence"] = round(float(sub_sim[best_sub]), 4)
    art["subcategory_scores"]     = {
        label: round(float(sub_sim[j]), 4)
        for j, label in enumerate(sub_info["labels"])
    }


print("\n── Level 1 — Category Distribution ──────────────────")
from collections import Counter
for cat, count in Counter(a["category"] for a in articles).items():
    pct = count / len(articles) * 100
    print(f"  {cat:<16} {count:>4}  ({pct:.0f}%)")

print("\n── Level 2 — Subcategory Distribution ───────────────")
for cat in cat_labels:
    sub_arts = [a for a in articles if a["category"] == cat]
    if not sub_arts: continue
    print(f"\n  [{cat}] ({len(sub_arts)} articles)")
    for sub, count in Counter(a["subcategory"] for a in sub_arts).most_common():
        avg_conf = np.mean([a["subcategory_confidence"] for a in sub_arts if a["subcategory"] == sub])
        print(f"    {sub:<40} {count:>4} articles  (avg conf: {avg_conf:.3f})")

print("\n── Sample (category + subcategory) ──────────────────")
shown = set()
for art in articles:
    key = (art["category"], art["subcategory"])
    if key not in shown and len(shown) < 6:
        shown.add(key)
        print(f"  [{art['category']}] → [{art['subcategory']}] ({art['subcategory_confidence']:.3f})")
        print(f"    {art['title'][:80]}")

Loading BAAI/bge-small-en-v1.5...
✅ Model loaded (dim=384)

Level 1 anchors encoded: (3, 384)
  [Data-level] subcategory anchors: (7, 384)
  [Model-level] subcategory anchors: (8, 384)
  [System-level] subcategory anchors: (7, 384)

Encoding 750 articles...
✅ Encoded in 33.1s

── Level 1 — Category Distribution ──────────────────
  Model-level       518  (69%)
  System-level      112  (15%)
  Data-level        120  (16%)

── Level 2 — Subcategory Distribution ───────────────

  [Data-level] (120 articles)
    Feature Extraction                         41 articles  (avg conf: 0.703)
    Edge Computing Frameworks                  36 articles  (avg conf: 0.690)
    Privacy & Data Security                    12 articles  (avg conf: 0.660)
    Data Quantization                           9 articles  (avg conf: 0.713)
    Synthetic & Augmented Data                  9 articles  (avg conf: 0.692)
    Data Aggregation                            8 articles  (avg conf: 0.711)
    Data Filtering   

## Cell 5 — Source-Aware Summary Generation with Ollama

Different sources need different prompt styles:
- **Semantic Scholar / arXiv**: rich abstract → ask for the *technical contribution*
- **Medium**: short preview → ask to *infer* the likely topic from title
- **LinkedIn**: conversational post → ask to extract the *core claim or insight*

A fallback returns the original summary if Ollama is unavailable.

In [22]:
import re, time
import ollama

OLLAMA_MODEL = "phi3:mini"  # run `ollama list` to confirm
SKIP_BELOW_CONFIDENCE = 0.65

# ── FEW-SHOT EXAMPLES ─────────────────────────────────────────────────────────
# Shown to the model in every prompt to anchor the expected output style.
# Deliberately cover all 3 categories and different formulations.

FEW_SHOT = """
EXAMPLES OF GOOD summaries (do exactly this):
  ✅ "INT4 quantization of Phi-3 achieves 94% accuracy retention with 3x memory reduction on Raspberry Pi."
  ✅ "Structured pruning removes 60% of parameters from LLaMA while preserving benchmark performance."
  ✅ "On-device NPU scheduling cuts transformer inference latency by 40% on mobile SoCs."
  ✅ "Federated learning with differential privacy enables SLM personalization without exposing user data."
  ✅ "Synthetic instruction data generated from GPT-4 reduces fine-tuning data needs by 80% for domain SLMs."
  ✅ "SLMs replace cloud LLMs in enterprise pipelines by cutting inference cost 10x with equivalent accuracy."

EXAMPLES OF BAD summaries (never do this):
  ❌ "This paper proposes a new method for..." ← starts with 'This paper'
  ❌ "This study investigates the use of..." ← starts with 'This study'
  ❌ "The authors present a framework that..." ← starts with 'The authors'
  ❌ "This article explores..." ← starts with 'This article'
  ❌ "In this work, we propose..." ← academic boilerplate
"""

BASE_INSTRUCTION = (
    "You are a technical analyst specializing in Edge AI and Small Language Models.\n"
    "Write exactly ONE sentence (max 25 words) capturing the key technical insight.\n\n"
    + FEW_SHOT +
    "\nRULES:\n"
    "- Start directly with the WHAT (technique, result, approach)\n"
    "- Include a specific number or metric if mentioned in the text\n"
    "- Use technical vocabulary (model names, method names, metrics)\n"
    "- Never start with: This/The paper/study/article/work/approach\n"
    "- One sentence only. No preamble, no explanation.\n\n"
)


def build_prompt(art: dict) -> str:
    source   = art.get("source", "")
    title    = art.get("title", "")
    summary  = art.get("summary", "")
    category = art.get("category", "")
    subcat   = art.get("subcategory", "")

    # The category context helps the model focus on the right aspect
    context = f"Classification context: {category} › {subcat}\n"

    if source in ("arxiv", "semantic_scholar"):
        return (
            f"{BASE_INSTRUCTION}"
            f"{context}"
            f"Research paper:\n"
            f"Title: {title}\n"
            f"Abstract: {summary}\n\n"
            f"One-sentence technical summary:\n"
        )
    elif source == "google_scholar":
        snippet = f"\nKeyword context: {summary}" if summary else ""
        return (
            f"{BASE_INSTRUCTION}"
            f"{context}"
            f"Research paper title: {title}{snippet}\n\n"
            f"One-sentence technical summary based on the title:\n"
        )
    elif source == "linkedin":
        return (
            f"{BASE_INSTRUCTION}"
            f"{context}"
            f"LinkedIn post:\n"
            f"Topic: {title}\n"
            f"Content: {summary}\n\n"
            f"Core technical insight in one sentence:\n"
        )
    else:  # medium + fallback
        preview = f"\nPreview: {summary}" if summary else ""
        return (
            f"{BASE_INSTRUCTION}"
            f"{context}"
            f"Article: {title}{preview}\n\n"
            f"One-sentence technical summary:\n"
        )


def clean_ai_output(text: str) -> str:
    """Post-process phi3 output to remove common artifacts."""
    text = text.strip().strip('"\'')
    # Remove leading label artifacts
    text = re.sub(
        r'^(Summary|Key insight|Technical insight|Contribution|One-sentence|Answer)[:.]\s*',
        '', text, flags=re.IGNORECASE
    ).strip()
    # If model still starts with banned phrases, strip the opener
    banned_openers = [
        r'^This (paper|study|article|work|research|approach)\s+\w+s\s+',
        r'^The (authors?|researchers?|paper|study)\s+\w+s?\s+',
        r'^In this (paper|work|study),?\s+',
        r'^We (propose|present|introduce|show|demonstrate)\s+',
    ]
    for pattern in banned_openers:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE).strip()
    # Capitalize first letter
    if text:
        text = text[0].upper() + text[1:]
    return text


# ── CHECK OLLAMA ──────────────────────────────────────────────────────────────
OLLAMA_AVAILABLE = False
try:
    available = [m['model'] for m in ollama.list().get('models', [])]
    print(f"✅ Ollama — installed models: {available}")
    if OLLAMA_MODEL not in available:
        prefix = OLLAMA_MODEL.split(':')[0]
        match  = [m for m in available if m.startswith(prefix)]
        if match:
            OLLAMA_MODEL = match[0]
            print(f"   Auto-adjusted → {OLLAMA_MODEL}")
            OLLAMA_AVAILABLE = True
        else:
            print(f"   ❌ Not found. Run: ollama pull {OLLAMA_MODEL}")
    else:
        OLLAMA_AVAILABLE = True
        print(f"   Using: {OLLAMA_MODEL}")
except Exception as e:
    print(f"⚠️  Ollama not reachable: {e}")


# ── GENERATE SUMMARIES ────────────────────────────────────────────────────────
to_process = [a for a in articles if a["category_confidence"] >= SKIP_BELOW_CONFIDENCE]
to_skip    = [a for a in articles if a["category_confidence"] <  SKIP_BELOW_CONFIDENCE]
for art in to_skip:
    art["ai_summary"] = trim_at_sentence(art.get("summary") or art.get("title", ""), 200)

print(f"\nOllama: {len(to_process)} articles | Fallback: {len(to_skip)}")
if OLLAMA_AVAILABLE:
    est = len(to_process) * 5
    print(f"Estimated time: ~{est//60}min {est%60}s on CPU")

total_time = 0
for i, art in enumerate(to_process):
    if not OLLAMA_AVAILABLE:
        art["ai_summary"] = trim_at_sentence(art.get("summary") or art.get("title", ""), 200)
        continue
    try:
        t0 = time.time()
        resp = ollama.chat(
            model=OLLAMA_MODEL,
            messages=[{"role": "user", "content": build_prompt(art)}],
            options={"temperature": 0.2, "num_predict": 60, "top_p": 0.9}
        )
        ai = clean_ai_output(resp["message"]["content"])
        art["ai_summary"] = ai
        elapsed = time.time() - t0
        total_time += elapsed
        flag = "✅" if not art["needs_review"] else "⚠️"
        print(f"{flag} [{i+1:03d}/{len(to_process)}] [{art.get('source','?')[:6]:<6}] "
              f"{art['subcategory'][:25]:<25} {elapsed:.1f}s | {ai[:70]}")
    except Exception as e:
        art["ai_summary"] = trim_at_sentence(art.get("summary") or art.get("title", ""), 200)
        print(f"❌ [{i+1:03d}] {e}")

if OLLAMA_AVAILABLE and total_time > 0:
    print(f"\n✅ Done — {total_time:.0f}s total, {total_time/len(to_process):.1f}s/article avg")

✅ Ollama — installed models: ['phi3:mini', 'minimax-m2.5:cloud', 'llama3.1:latest']
   Using: phi3:mini

Ollama: 648 articles | Fallback: 102
Estimated time: ~54min 0s on CPU


1it [00:15, 15.18s/it]

✅ [001/648] [google] Energy-efficient Model De 15.2s | CoDi-ABEM leverages prompt compression and knowledge distillation to e


2it [00:27, 13.27s/it]

✅ [002/648] [google] Edge & IoT Deployment     11.9s | Adaptive quantization techniques in neural networks reduce model size 


...

✅ [224/648] [google] Model Quantization        10.6s | Experiments demonstrate that compressing Phi-3 to a 3-bit representati


225it [1:45:55, 11.91s/it]

✅ [225/648] [google] Synthetic & Augmented Dat 12.1s | Knowledge distillation transfers knowledge effectively with up to 90% 


225it [1:45:59, 28.27s/it]


KeyboardInterrupt: 

## Cell 6 — Validate, Inspect & Save

Final quality checks before saving. Pay attention to:
1. Category distribution per source — LinkedIn may skew System-level (deployment talk)
2. Low-confidence articles — candidates for anchor tuning
3. The final JSON schema before plugging into app.py

In [ ]:
# ── GLOBAL DISTRIBUTION ───────────────────────────────────────────────────────
print("═" * 55)
print("ENRICHMENT SUMMARY")
print("═" * 55)

print(f"\nTotal articles enriched: {len(articles)}")

print("\n── Global Category Distribution ──")
for cat, count in Counter(a["category"] for a in articles).items():
    pct = count / len(articles) * 100
    bar = "█" * count
    print(f"  {cat:<16} {count:>4} ({pct:.0f}%)  {bar}")

# ── CROSS-TABLE: Source × Category ───────────────────────────────────────────
print("\n── Source × Category Matrix ──────────────────────────")
sources = sorted(set(a.get("source") for a in articles))
cats    = ["Data-level", "Model-level", "System-level"]
header  = f"{'Source':<20}" + "".join(f"{c:<16}" for c in cats) + "Flagged"
print(header)
print("-" * len(header))
for src in sources:
    src_arts = [a for a in articles if a.get("source") == src]
    row = f"{src:<20}"
    for cat in cats:
        n = sum(1 for a in src_arts if a["category"] == cat)
        row += f"{n:<16}"
    flagged = sum(1 for a in src_arts if a.get("needs_review"))
    row += str(flagged)
    print(row)

# ── CONFIDENCE STATS PER SOURCE ───────────────────────────────────────────────
print("\n── Confidence Stats per Source ───────────────────────")
for src in sources:
    confs = [a["category_confidence"] for a in articles if a.get("source") == src]
    if confs:
        print(f"  {src:<20} min={min(confs):.3f}  avg={np.mean(confs):.3f}  max={max(confs):.3f}")

# ── SAMPLE ENRICHED ARTICLE PER SOURCE ───────────────────────────────────────
print("\n── Sample Enriched Article per Source ───────────────")
shown = set()
for art in articles:
    src = art.get("source")
    if src not in shown:
        shown.add(src)
        display_fields = {
            "source":              art.get("source"),
            "title":               art.get("title", "")[:80],
            "category":            art.get("category"),
            "category_confidence": art.get("category_confidence"),
            "needs_review":        art.get("needs_review"),
            "ai_summary":          art.get("ai_summary", "")[:120],
        }
        print(f"\n{json.dumps(display_fields, indent=2)}")

# ── SAVE ──────────────────────────────────────────────────────────────────────
Path("data").mkdir(exist_ok=True)

# Final field ordering for clean JSON
FIELD_ORDER = [
    "title", "authors", "summary", "published_date", "url", "source",
    "citation_text", "input_text",
    "category", "category_confidence", "category_scores", "needs_review",
    "ai_summary"
]

def reorder(art: dict) -> dict:
    ordered = {k: art[k] for k in FIELD_ORDER if k in art}
    # Keep any extra fields that might exist
    for k, v in art.items():
        if k not in ordered:
            ordered[k] = v
    return ordered

output = [reorder(a) for a in articles]

out_path = Path("../data/enriched_articles.json")
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"\n✅ Saved → {out_path}")
print(f"   {len(output)} articles | {out_path.stat().st_size / 1024:.1f} KB")

═══════════════════════════════════════════════════════
ENRICHMENT SUMMARY
═══════════════════════════════════════════════════════

Total articles enriched: 750

── Global Category Distribution ──
  Model-level       518 (69%)  ██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  System-level      112 (15%)  ████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  Data-level        120 (16%)  ██████████████████████████████████████████████████████████████████████████████